# K-Nearest Neighbors Classifier – Practice Skeleton

**Short name (GitHub):** `KNN_Cancer`  
**Lab source:** Codecademy *K-Nearest Neighbors Classifier* (movies from-scratch) + *Cancer Classifier* (sklearn)  
**Language:** Python (NumPy + pandas + Matplotlib + scikit-learn)

Use this notebook to practice. Open **`KNN_Cancer_Solution.ipynb`** only after you attempt each exercise.  
Companion files: `KNN_Cancer_Cheatsheet.docx`, `KNN_Cancer_Reusable_Template.ipynb`, `knn_cancer_flowchart.png`, `KNN_Cancer.py`.

### Learning objectives
- Compute Euclidean distance in 2-D and in *n*-D
- Min-max normalize so budget does not drown year / duration
- Find the *k* nearest labeled movies and majority-vote good vs bad
- Load the Wisconsin breast-cancer set, split train/valid, fit `KNeighborsClassifier`
- Sweep *k* and plot validation accuracy (bias–variance)
- Alternate implementations (broadcast NumPy, Manhattan, `NearestNeighbors`)
- Extra practice on a 2-D blob set and a DTI/utilization loan book
- Monte-Carlo: *k*, label noise, sample size, extra noise features
- Rewrite the same result for an analyst, a medical director, a patient, a non-specialist

### Data files
- `data/knn_movies.csv` — 40 films (duration, year, budget, good)
- `data/knn_cancer.csv` — 569 cells × 30 features + target (0 = malignant, 1 = benign)
- `data/knn_2d.csv` — synthetic 2-feature set for boundaries
- `data/knn_loans.csv` — DTI + utilization → default

### Flowchart
Open `knn_cancer_flowchart.png` while you work.


## Inline cheat-sheet (keep this cell visible)

See also **`KNN_Cancer_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Euclidean | $d(a,b)=\sqrt{\sum_j (a_j-b_j)^2}$ |
| Manhattan | $d_1(a,b)=\sum_j \|a_j-b_j\|$ |
| Min-max | $x'=(x-x_{\min})/(x_{\max}-x_{\min})$ |
| Vote | predict the majority class among the $k$ nearest |
| Ties | prefer odd $k$, else use the closest neighbor |
| Overfit | $k$ too small → outliers dominate |
| Underfit | $k$ too large → vote ≈ global majority |
| sklearn | `KNeighborsClassifier(n_neighbors=k).fit(X,y).score(Xv,yv)` |
| Split | `train_test_split(..., test_size=0.2, random_state=100, stratify=y)` |
| Cancer labels | `target` 0 = malignant, 1 = benign |

**Flow:** features → scale → split → distance → $k$ neighbors → vote → sweep $k$ → simulate.


## 0. Packages


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix

%matplotlib inline
np.set_printoptions(precision=6, suppress=True)
print("Libraries loaded")


## 1. Distance between points (2-D)

Two movies as `[runtime_minutes, release_year]`.

$$
d = \sqrt{(L_1-L_2)^2 + (Y_1-Y_2)^2}
$$

### Task 1.1 — `distance_2d`
Write `distance_2d(movie1, movie2)` that returns the Euclidean distance. Use `** 0.5` or `math.sqrt`.


In [ ]:
star_wars = [125, 1977]
raiders = [115, 1981]
mean_girls = [97, 2004]

def distance_2d(movie1, movie2):
    # YOUR CODE HERE
    pass

print(distance_2d(star_wars, raiders))
print(distance_2d(star_wars, mean_girls))
# Which film is closer to Star Wars?


## 2. Distance in *n* dimensions

$$
d(A,B)=\sqrt{\sum_{j=1}^{n}(A_j-B_j)^2}
$$

### Task 2.1 — generalize with a loop


In [ ]:
star_wars_3 = [125, 1977, 11_000_000]
raiders_3 = [115, 1981, 18_000_000]
mean_girls_3 = [97, 2004, 17_000_000]

def distance(a, b):
    """Euclidean distance for lists / 1-D arrays of any length."""
    # YOUR CODE HERE
    pass

print(distance(star_wars_3, raiders_3))
print(distance(star_wars_3, mean_girls_3))


### Task 2.2 — NumPy alternate (no Python loop)

`np.sqrt(np.sum((np.asarray(a)-np.asarray(b))**2))` or `np.linalg.norm`.


In [ ]:
def distance_np(a, b):
    # YOUR CODE HERE
    pass

print(distance_np(star_wars_3, raiders_3))
print("match loop?", np.isclose(distance(star_wars_3, raiders_3), distance_np(star_wars_3, raiders_3)))


## 3. Min-max normalization

Budget is millions of dollars; year spans ~100. Without scaling, budget owns the distance.

$$
x' = \frac{x - x_{\min}}{x_{\max}-x_{\min}}
$$

### Task 3.1 — `min_max_normalize(lst)`


In [ ]:
release_dates = [1897.0, 1998.0, 2000.0, 1948.0, 1962.0,
                 1950.0, 1975.0, 1960.0, 2017.0, 1937.0]

def min_max_normalize(lst):
    # YOUR CODE HERE
    pass

print(min_max_normalize(release_dates))
# What does 1897 become? Why is it near 0?


## 4. Movie data + from-scratch classifier

### Task 4.1 — load `data/knn_movies.csv`
Build `movie_dataset` as `{title: [duration, year, budget]}` and `movie_labels` as `{title: good}`.


In [ ]:
# YOUR CODE HERE
movies = None
movie_dataset = {}
movie_labels = {}

print("n movies:", len(movie_dataset))
print("sample:", list(movie_dataset.items())[:2])


### Task 4.2 — column-wise min-max on the three features
Normalize every movie in-place (or into a new dict `movie_dataset_n`). Store column mins/maxes so a new point can use the same scale.


In [ ]:
def fit_minmax(dataset):
    """Return (mins, maxs) arrays of shape (n_features,)."""
    # YOUR CODE HERE
    pass

def apply_minmax(point, mins, maxs):
    # YOUR CODE HERE
    pass

def normalize_dataset(dataset):
    # YOUR CODE HERE
    pass

mins, maxs = None, None
movie_dataset_n = {}
print("mins:", mins)
print("maxs:", maxs)


### Task 4.3 — `classify(unknown, dataset, labels, k)`

1. For every title compute `[distance, title]`.
2. Sort ascending.
3. Take the first *k*.
4. Count good vs bad labels.
5. Return 1 if good wins, else 0.


In [ ]:
def classify(unknown, dataset, labels, k):
    # YOUR CODE HERE
    pass


### Task 4.4 — classify a new film

*Call Me By Your Name*: budget 3_500_000, runtime 132, year 2017. Confirm the title is **not** already in the dict, normalize, predict with *k* = 5.


In [ ]:
print("already in set?", "Call Me By Your Name" in movie_dataset)  # may be True in our csv
my_movie = [132, 2017, 3_500_000]   # duration, year, budget — same order as the dict
# If the title is already in the training dict, drop it before classifying (no self-match).
# YOUR CODE HERE


## 5. Alternate neighbor search

### Task 5.1 — broadcast all pairwise distances
Stack the dataset into an array `X` of shape `(m, n)` and compute `np.linalg.norm(X - unknown, axis=1)`.


In [ ]:
def classify_np(unknown, X, y, k):
    """unknown: (n,), X: (m,n), y: (m,) of 0/1."""
    # YOUR CODE HERE
    pass


### Task 5.2 — Manhattan distance variant


In [ ]:
def classify_manhattan(unknown, X, y, k):
    # YOUR CODE HERE
    pass


## 6. Breast-cancer data (sklearn project)

Wisconsin Diagnostic Breast Cancer: 569 fine-needle aspirates, 30 numeric features, binary target.

### Task 6.1 — load from csv **or** `load_breast_cancer()`
Print `feature_names[:8]`, first row, `target[:10]`, `target_names`.  
Was the first point malignant or benign?


In [ ]:
# YOUR CODE HERE
breast_cancer_data = None
print("use either sklearn loader or data/knn_cancer.csv")


### Task 6.2 — class balance
How many malignant (0) vs benign (1)?


In [ ]:
# YOUR CODE HERE


## 7. Train / validation split

### Task 7.1
`train_test_split` on the features and labels.

- `test_size = 0.2`
- `random_state = 100` (Codecademy seed)
- `stratify = y` so both classes stay in the same ratio


In [ ]:
# YOUR CODE HERE
training_data = validation_data = training_labels = validation_labels = None
print("train", None, "valid", None)


## 8. `KNeighborsClassifier`

### Task 8.1 — *k* = 3, unscaled
Fit, then `score` on the validation set.


In [ ]:
# YOUR CODE HERE
classifier = None


### Task 8.2 — same *k* after `MinMaxScaler`
Fit the scaler on **training data only**, transform both splits.


In [ ]:
# YOUR CODE HERE


## 9. Sweep *k* and graph

### Task 9.1
For `k` in 1 … 50 fit a scaled KNN and store validation accuracy. Print the *k* that maximizes it.


In [ ]:
k_list = list(range(1, 51))
accuracies = []
# YOUR CODE HERE

print("best k, acc:", None, None)


### Task 9.2 — plot
x = `k_list`, y = `accuracies`. Labels: `"k"`, `"Validation Accuracy"`, title `"Breast Cancer Classifier Accuracy"`.


In [ ]:
# YOUR CODE HERE
plt.xlabel("k")
plt.ylabel("Validation Accuracy")
plt.title("Breast Cancer Classifier Accuracy")
plt.show()


## 10. More practice

### Task 10.1 — 2-D blobs (`data/knn_2d.csv`)
Split 80/20, plot the two classes, fit *k* = 1 and *k* = 21, sketch (or contour) the two decision regions. Which *k* looks overfit?


In [ ]:
# YOUR CODE HERE


### Task 10.2 — loan book (`data/knn_loans.csv`)
Features `dti`, `utilization`; label `default`. Scale, split, report validation accuracy at *k* = 1, 5, 15 and a confusion matrix for the best of those three.


In [ ]:
# YOUR CODE HERE


## 11. Simulation (edit the box, re-run)

Change any of `K_FIXED`, `NOISE`, `TRAIN_FRAC`, `N_EXTRA` and re-run the cell.  
You should see accuracy move the way the four-panel figure `knn_cancer_simulation.png` describes.


In [ ]:
# ----- editable -----
K_FIXED = 15          # neighbors
NOISE = 0.00          # train-label flip probability
TRAIN_FRAC = 1.00     # fraction of the training split to keep
N_EXTRA = 0           # extra Gaussian noise columns (curse of dimensionality)
RANDOM_STATE = 100
# --------------------

bc = load_breast_cancer()
X, y = bc.data, bc.target
Xtr, Xva, ytr, yva = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
sc = MinMaxScaler()
Xtr = sc.fit_transform(Xtr)
Xva = sc.transform(Xva)

rng = np.random.default_rng(RANDOM_STATE)
m = max(K_FIXED + 1, int(len(Xtr) * TRAIN_FRAC))
idx = rng.choice(len(Xtr), size=m, replace=False)
Xtr, ytr = Xtr[idx], ytr[idx].copy()
flip = rng.random(len(ytr)) < NOISE
ytr[flip] = 1 - ytr[flip]
if N_EXTRA > 0:
    Xtr = np.hstack([Xtr, rng.normal(size=(len(Xtr), N_EXTRA))])
    Xva = np.hstack([Xva, rng.normal(size=(len(Xva), N_EXTRA))])
    sc2 = MinMaxScaler()
    Xtr = sc2.fit_transform(Xtr)
    Xva = sc2.transform(Xva)

clf = KNeighborsClassifier(n_neighbors=min(K_FIXED, len(Xtr)))
clf.fit(Xtr, ytr)
acc = clf.score(Xva, yva)
print(f"valid acc = {acc:.4f}   (k={K_FIXED}, noise={NOISE}, n_train={len(Xtr)}, extra={N_EXTRA})")
print("try: K_FIXED=1, NOISE=0.2, TRAIN_FRAC=0.2, N_EXTRA=80")


## 12. Audience rewrite

Using the attached audience notes (data literacy, subject knowledge, experts / technicians / executives / nonspecialists), write **four 3–5 sentence** versions of:

> “On a held-out 20% of the Wisconsin cells, a min-max-scaled KNN with k ≈ 15 reached about 97% validation accuracy. k = 1 is twitchier; stuffing the matrix with noise features hurts distance.”

1. Quant / research analyst  
2. Medical director / tumor-board chair  
3. Patient-education leaflet  
4. Non-specialist friend  

Keep numbers, drop jargon that each audience does not need.


In [ ]:
# Write your four paragraphs as strings, then print them.
analyst = """..."""
director = """..."""
patient = """..."""
friend = """..."""
print(analyst); print(); print(director); print(); print(patient); print(); print(friend)


## Done

Compare with `KNN_Cancer_Solution.ipynb`. Reuse the pattern on a new table via `KNN_Cancer_Reusable_Template.ipynb`.
